# Day 1 — Data Ingestion & Exploration

This notebook demonstrates the complete Day 1 pipeline:
1. Copy all 10 source CSVs into `data/raw/`
2. Profile each dataset (shape, dtypes, nulls, duplicates, sample rows)
3. Explore `01_fund_master.csv` (unique AMCs, categories, sub-categories, risk grades)
4. Fetch live NAV data from `api.mfapi.in` for targeted AMFI codes
5. Cross-dataset AMFI code validation between fund master and NAV history

In [ ]:
import pandas as pd
import numpy as np
import os
import shutil
from pathlib import Path
from datetime import datetime

BASE_DIR = Path().resolve().parent
SOURCE_DIR = BASE_DIR / "data_sets"
RAW_DIR = BASE_DIR / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

print(f"Base directory : {BASE_DIR}")
print(f"Source datasets: {SOURCE_DIR}")
print(f"Raw output     : {RAW_DIR}")

## Step 1 — Copy Source CSVs to `data/raw/`

In [ ]:
csv_files = sorted(SOURCE_DIR.glob("*.csv"))
print(f"Found {len(csv_files)} source CSV files:\n")

for src in csv_files:
    dst = RAW_DIR / src.name
    if not dst.exists():
        shutil.copy(src, dst)
        print(f"  ✅ Copied {src.name}")
    else:
        print(f"  ⏭️  {src.name} already exists in data/raw/")

## Step 2 — Profile All 10 Datasets

For each CSV we print: **shape**, **column dtypes**, **null counts**, **duplicate rows**, and **sample rows (head)**.

In [ ]:
raw_files = sorted(RAW_DIR.glob("[0-9]*.csv"))

profile_summary = []

for fp in raw_files:
    df = pd.read_csv(fp)
    nulls = df.isnull().sum()
    total_nulls = nulls.sum()
    dups = df.duplicated().sum()

    profile_summary.append({
        "Dataset": fp.name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Nulls": total_nulls,
        "Duplicates": dups,
    })

    print(f"{'=' * 70}")
    print(f"📄 {fp.name}")
    print(f"{'=' * 70}")
    print(f"  Shape       : {df.shape}")
    print(f"  Columns     : {list(df.columns)}")
    print(f"  Total Nulls : {total_nulls}")
    print(f"  Duplicates  : {dups}")
    if total_nulls > 0:
        print(f"  Null detail : {nulls[nulls > 0].to_dict()}")
    print(f"\n  Dtypes:")
    for col, dtype in df.dtypes.items():
        print(f"    {col:40s} {dtype}")
    print(f"\n  Head (5 rows):")
    display(df.head())
    print()

In [ ]:
# Summary table
summary_df = pd.DataFrame(profile_summary)
print("\n📊 INGESTION PROFILE SUMMARY")
print("=" * 70)
display(summary_df)
print(f"\nTotal records across all datasets: {summary_df['Rows'].sum():,}")
print(f"Total null values: {summary_df['Nulls'].sum():,}")
print(f"Total duplicates:  {summary_df['Duplicates'].sum():,}")

## Step 3 — Fund Master Exploration

Extract unique fund houses, categories, sub-categories, and risk grades from `01_fund_master.csv`.

In [ ]:
master = pd.read_csv(RAW_DIR / "01_fund_master.csv")
print(f"Total Schemes: {len(master)}")
display(master.head())

In [ ]:
print(f"\n🏢 Unique Fund Houses ({master['fund_house'].nunique()}):")
for fh in sorted(master['fund_house'].unique()):
    print(f"  • {fh}")

print(f"\n📁 Unique Categories ({master['category'].nunique()}):")
for cat in sorted(master['category'].unique()):
    print(f"  • {cat}")

print(f"\n📂 Unique Sub-Categories ({master['sub_category'].nunique()}):")
for sub in sorted(master['sub_category'].unique()):
    print(f"  • {sub}")

print(f"\n⚠️ Unique Risk Categories ({master['risk_category'].nunique()}):")
for rc in sorted(master['risk_category'].unique()):
    print(f"  • {rc}")

print(f"\nTotal AMFI Codes: {master['amfi_code'].nunique()}")

## Step 4 — Live NAV Fetch from mfapi.in

Fetch live NAV data for 6 target mutual fund schemes via the public `api.mfapi.in` REST API.

In [ ]:
import requests

TARGETS = {
    "125497": "HDFC_Top_100_Direct",
    "119551": "SBI_Bluechip",
    "120503": "ICICI_Bluechip",
    "118632": "Nippon_Large_Cap",
    "119092": "Axis_Bluechip",
    "120841": "Kotak_Bluechip",
}

for amfi_code, label in TARGETS.items():
    url = f"https://api.mfapi.in/mf/{amfi_code}"
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        nav_data = data.get("data", [])
        if nav_data:
            df_nav = pd.DataFrame(nav_data)
            out_path = RAW_DIR / f"raw_nav_{amfi_code}.csv"
            df_nav.to_csv(out_path, index=False)
            print(f"  ✅ {label} ({amfi_code}): {len(df_nav)} NAV records saved")
        else:
            print(f"  ⚠️  {label} ({amfi_code}): No NAV data returned")
    except Exception as e:
        print(f"  ❌ {label} ({amfi_code}): {e}")

## Step 5 — AMFI Code Cross-Validation

Verify that all AMFI codes in the NAV history match those in the fund master.

In [ ]:
nav = pd.read_csv(RAW_DIR / "02_nav_history.csv")

master_codes = set(master['amfi_code'].astype(str).unique())
nav_codes = set(nav['amfi_code'].astype(str).unique())

in_master_not_nav = master_codes - nav_codes
in_nav_not_master = nav_codes - master_codes

print(f"AMFI codes in fund_master : {len(master_codes)}")
print(f"AMFI codes in nav_history : {len(nav_codes)}")
print(f"In master but NOT in NAV  : {len(in_master_not_nav)} → {in_master_not_nav or 'None'}")
print(f"In NAV but NOT in master  : {len(in_nav_not_master)} → {in_nav_not_master or 'None'}")

if not in_master_not_nav and not in_nav_not_master:
    print("\n✅ AMFI code integrity check PASSED — all codes match.")
else:
    print("\n⚠️  AMFI code mismatch detected — review before loading into DB.")

---
*Day 1 Data Ingestion complete. All 10 datasets profiled, live NAV data fetched, and AMFI code integrity validated.*